# Per-Tier Demand Curves from ClusterData 2019 (standalone)

**Purpose:** Extract the **real** per-tier aggregate demand curves — non-deferrable **service** vs deferrable **batch** — for cells a–d, at 5-minute resolution, from `instance_usage`.

This is the **ground-truth** replacement for the local approximation in `scripts/derive_tier_curves.py` (which spreads job-level *requests* over submit→end windows; those windows include queueing delay and requests ≠ usage). Here we split the *measured usage* itself by tier.

**Tier definition** (Tirmazi et al. 2020, *"Borg: the Next Generation"*, EuroSys '20, §2): deferrable **batch** = the no-SLO tiers — free (`priority <= 99`) OR best-effort batch (`priority BETWEEN 110 AND 115`). Everything else (mid / production / monitoring) is **service**.

**Why this organization:** mirrors Radovanović et al. (2023), *"Carbon-Aware Computing for Datacenters"* (CICS), which consumes real aggregate flexible vs inflexible demand curves per cluster — no synthetic workload generation in the loop (see `thesis_overview.md` §1.2, §3.8, §7.10).

**Output:** `data/cells/cell_{a..d}_tiers.csv` with columns `timestep, cpu_demand_norm, batch_share, service_demand_norm, batch_demand_norm` — drop-in replacement for the locally derived files (env reads them via the `tier_curves` entry in the scenario YAMLs).

**This notebook is standalone** — you do *not* need to run `extract_clusterdata2019_full.ipynb`. Filters replicate that notebook's Dataset 1 exactly (top-level instances only, >=5-min usage records), so `service + batch` reproduces `cells/cell_X.csv`.

> ⚠ If running through the VS Code Jupyter extension, `files.download()` does nothing — grab the zip from the Colab **Files panel** (left sidebar) or copy it to Drive.

In [ ]:
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'aeee-thesis'  # <-- your GCP project with BigQuery API enabled

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID)
print(f'Authenticated with project: {PROJECT_ID}')

In [ ]:
import os
import pandas as pd

os.makedirs('data/cells', exist_ok=True)
CELLS = ['a', 'b', 'c', 'd']

for cell in CELLS:
    print(f'--- Cell {cell}: per-tier usage curves ---')
    ds = f'`google.com:google-cluster-data`.clusterdata_2019_{cell}'
    # priority lives on collection_events, not instance_usage -> join on collection_id.
    # Filters replicate Dataset 1 of extract_clusterdata2019_full.ipynb exactly.
    query = f"""
    WITH cap AS (
        SELECT SUM(cpu_cap) AS cpu_capacity FROM (
            SELECT machine_id, MAX(capacity.cpus) AS cpu_cap
            FROM {ds}.machine_events GROUP BY 1
        )
    ),
    prio AS (
        SELECT collection_id, MAX(priority) AS priority
        FROM {ds}.collection_events
        WHERE type = 0
        GROUP BY collection_id
    )
    SELECT
        CAST(FLOOR(u.start_time / (1e6 * 300)) AS INT64) AS time_bucket,
        SUM(u.average_usage.cpus) / (SELECT cpu_capacity FROM cap) AS total_norm,
        SUM(IF(p.priority <= 99 OR p.priority BETWEEN 110 AND 115,
               u.average_usage.cpus, 0)) / (SELECT cpu_capacity FROM cap) AS batch_norm
    FROM {ds}.instance_usage u
    LEFT JOIN prio p USING (collection_id)
    WHERE (u.alloc_collection_id IS NULL OR u.alloc_collection_id = 0)
        AND (u.end_time - u.start_time) >= (5 * 60 * 1e6)
    GROUP BY 1
    ORDER BY 1
    """
    df = client.query(query).to_dataframe()
    df['timestep'] = df['time_bucket'] - df['time_bucket'].min()
    df['cpu_demand_norm'] = df['total_norm'].clip(0.0, 1.0)
    df['batch_demand_norm'] = df['batch_norm'].clip(0.0, 1.0).clip(upper=df['cpu_demand_norm'])
    df['service_demand_norm'] = df['cpu_demand_norm'] - df['batch_demand_norm']
    df['batch_share'] = (df['batch_demand_norm'] / df['cpu_demand_norm'].clip(lower=1e-9)).clip(0, 1)
    out = df[['timestep', 'cpu_demand_norm', 'batch_share',
              'service_demand_norm', 'batch_demand_norm']]
    path = f'data/cells/cell_{cell}_tiers.csv'
    out.to_csv(path, index=False)
    bf = out['batch_demand_norm'].sum() / out['cpu_demand_norm'].sum()
    pm = out['batch_demand_norm'].max() / out['batch_demand_norm'].mean()
    print(f'  {len(out)} rows -> {path}   batch_fraction={bf:.3f}  batch peak/mean={pm:.2f}')

print('\nDone — measured per-tier curves (service + batch = measured aggregate).')

In [ ]:
# Zip + download. In the VS Code Jupyter bridge this download() silently no-ops:
# use the Colab Files panel (left sidebar) -> tier_curves.zip -> Download instead.
import shutil
shutil.make_archive('tier_curves', 'zip', '.', 'data')
try:
    from google.colab import files
    files.download('tier_curves.zip')
except Exception as e:
    print(f'files.download unavailable ({e}); use the Files panel.')
print('Extract into C:\\Projects\\thesis\\ (overwrites the locally derived *_tiers.csv).')